# Companion Notebook — Regressão Linear

Este notebook acompanha as notas de aula sobre **covariância, correlação e regressão linear**.

A ideia é retirar os blocos de código da seção “Implementação em Python” das notas e transformá-los em uma atividade prática, na qual o estudante possa executar, alterar parâmetros, inspecionar saídas e interpretar resultados.

## Objetivos práticos

Ao final deste notebook, você deverá ser capaz de:

1. calcular covariância e correlação de Pearson em Python;
2. ajustar uma regressão linear simples com `statsmodels`;
3. interpretar coeficientes, $R^2$, testes $t$, teste $F$ e intervalos;
4. construir intervalos de confiança e intervalos de predição;
5. diagnosticar resíduos por meio de gráficos;
6. ajustar uma regressão linear múltipla;
7. calcular VIF para avaliar multicolinearidade.

## Como usar

Execute as células em ordem. Algumas variáveis, como `modelo`, são criadas em uma célula e reutilizadas nas seguintes.

As bibliotecas usadas são:

- `numpy`
- `pandas`
- `matplotlib`
- `scipy`
- `statsmodels`

Caso alguma biblioteca não esteja instalada, use:

```bash
pip install numpy pandas matplotlib scipy statsmodels
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

from scipy.stats import pearsonr
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Correlação de Pearson: idade e peso de crianças

O primeiro exemplo usa os mesmos dados das notas de aula: idade, em anos, e peso, em kg, de 6 crianças.

Vamos calcular a correlação de Pearson de duas maneiras:

1. manualmente, usando a definição baseada na covariância;
2. usando a função `pearsonr` da biblioteca `scipy`.

Em seguida, construiremos o diagrama de dispersão.

In [ ]:
# Dados: idade (x) e peso (y)
x = np.array([7, 6, 8, 5, 6, 9])
y = np.array([12, 8, 12, 10, 11, 13])

# Covariância amostral entre x e y
cov_xy = np.cov(x, y, ddof=1)[0, 1]

# Desvios-padrão amostrais
sx = np.std(x, ddof=1)
sy = np.std(y, ddof=1)

# Correlação calculada pela definição
r_manual = cov_xy / (sx * sy)

# Correlação via scipy
# pearsonr também retorna o p-valor do teste H0: rho = 0
r, p_valor = pearsonr(x, y)

print(f"Covariância amostral = {cov_xy:.4f}")
print(f"r calculado manualmente = {r_manual:.4f}")
print(f"r via scipy = {r:.4f}")
print(f"p-valor = {p_valor:.4f}")

In [ ]:
# Diagrama de dispersão
plt.figure(figsize=(5, 4))
plt.scatter(x, y, zorder=3)
plt.xlabel("Idade (anos)")
plt.ylabel("Peso (kg)")
plt.title(f"Diagrama de dispersão (r = {r:.3f})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Perguntas para interpretação

1. O sinal de $r$ é positivo ou negativo? O que isso indica?
2. A associação parece fraca, moderada ou forte?
3. O diagrama de dispersão confirma a interpretação numérica?
4. O p-valor deve ser interpretado com cautela neste exemplo? Por quê?

# 2. Regressão linear simples: horas de estudo e nota final

Agora ajustaremos o modelo:

\[
Y_i = \beta_0 + \beta_1 x_i + \varepsilon_i
\]

em que:

- $X$ = horas semanais de estudo;
- $Y$ = nota final.

O modelo será ajustado com `statsmodels`, que fornece uma tabela completa com coeficientes, erros padrão, estatísticas de teste, p-valores, $R^2$, $R^2$ ajustado e teste $F$ global.

In [ ]:
# Dados: horas de estudo (x) e nota final (y)
x = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10])
y = np.array([55, 60, 62, 68, 72, 75, 80, 83, 88])

# OLS com statsmodels
# sm.add_constant adiciona a coluna do intercepto
X = sm.add_constant(x)
modelo = sm.OLS(y, X).fit()

print(modelo.summary())

## Interpretação mínima esperada

Após executar a célula anterior, identifique:

1. o intercepto $\hat{\beta}_0$;
2. a inclinação $\hat{\beta}_1$;
3. o valor de $R^2$;
4. o p-valor associado à inclinação;
5. a equação ajustada.

No contexto do problema, a inclinação indica quantos pontos a nota média esperada aumenta para cada hora adicional de estudo por semana.

In [ ]:
# Extração programática dos principais resultados
b0, b1 = modelo.params
r2 = modelo.rsquared
r2_ajustado = modelo.rsquared_adj
s = np.sqrt(modelo.mse_resid)

print(f"Equação ajustada: y_hat = {b0:.3f} + {b1:.3f} x")
print(f"R² = {r2:.4f}")
print(f"R² ajustado = {r2_ajustado:.4f}")
print(f"Erro padrão da regressão S = {s:.4f}")

# 3. Previsão, intervalo de confiança e intervalo de predição

Para um novo valor $x^\*$, podemos calcular:

- a previsão média $\hat{y}^\*$;
- o intervalo de confiança para $E[Y \mid x^\*]$;
- o intervalo de predição para uma nova observação individual.

O intervalo de predição é mais largo porque inclui, além da incerteza sobre a média, a variabilidade natural de observações individuais ao redor da reta.

In [ ]:
# Previsão para um aluno que estuda 5.5 horas por semana
x_novo = np.array([[1, 5.5]])  # intercepto + x*
ic = modelo.get_prediction(x_novo).summary_frame(alpha=0.05)

ic[['mean', 'mean_ci_lower', 'mean_ci_upper',
    'obs_ci_lower', 'obs_ci_upper']]

In [ ]:
# Gráfico com reta ajustada, IC 95% para a média e IP 95% para observação individual
x_plot = np.linspace(x.min(), x.max(), 100)
X_plot = sm.add_constant(x_plot)

y_pred = modelo.predict(X_plot)
pred_full = modelo.get_prediction(X_plot)
ic_full = pred_full.summary_frame(alpha=0.05)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, zorder=4, label='Dados')
plt.plot(x_plot, y_pred, label='Reta ajustada')

plt.fill_between(
    x_plot,
    ic_full['mean_ci_lower'],
    ic_full['mean_ci_upper'],
    alpha=0.25,
    label='IC 95% para a média'
)

plt.fill_between(
    x_plot,
    ic_full['obs_ci_lower'],
    ic_full['obs_ci_upper'],
    alpha=0.10,
    label='IP 95% para observação individual'
)

plt.xlabel('Horas de estudo')
plt.ylabel('Nota')
plt.legend()
plt.tight_layout()
plt.show()

## Perguntas para interpretação

1. O intervalo de predição ficou mais largo do que o intervalo de confiança?
2. O que significa prever a nota média de alunos com 5,5 horas de estudo?
3. O que significa prever a nota de um aluno individual com 5,5 horas de estudo?
4. Por que não seria adequado usar esse modelo para prever a nota de um aluno que estuda 25 horas por semana?

# 4. Diagnóstico de resíduos

A validade das inferências depende das suposições do modelo. Por isso, depois de ajustar uma regressão, devemos examinar os resíduos.

Nesta seção, construiremos:

1. gráfico de resíduos vs. valores ajustados;
2. gráfico Normal Q-Q dos resíduos;
3. gráfico de resíduos padronizados vs. valores ajustados.

O objetivo não é “provar” que as suposições são verdadeiras, mas procurar indícios fortes de violação.

In [ ]:
residuos = modelo.resid
residuos_padronizados = modelo.get_influence().resid_studentized_internal
valores_ajustados = modelo.fittedvalues

plt.figure(figsize=(6, 4))
plt.scatter(valores_ajustados, residuos)
plt.axhline(0, linestyle='--', linewidth=1)
plt.xlabel('Valores ajustados')
plt.ylabel('Resíduos')
plt.title('Resíduos vs. Valores Ajustados')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
stats.probplot(residuos, dist='norm', plot=plt)
plt.title('Normal Q-Q dos Resíduos')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(valores_ajustados, residuos_padronizados)
plt.axhline(0, linestyle='--', linewidth=1)
plt.axhline(2, linestyle=':', linewidth=1)
plt.axhline(-2, linestyle=':', linewidth=1)
plt.xlabel('Valores ajustados')
plt.ylabel('Resíduos padronizados')
plt.title('Resíduos Padronizados vs. Valores Ajustados')
plt.tight_layout()
plt.show()

## Perguntas para diagnóstico

1. Há padrão curvo no gráfico de resíduos?
2. A variância dos resíduos parece aumentar ou diminuir com os valores ajustados?
3. Há resíduos padronizados maiores que 2 em valor absoluto?
4. O gráfico Q-Q sugere normalidade aproximada dos resíduos?

# 5. Regressão linear múltipla e VIF

Agora ajustaremos um modelo com mais de um preditor:

\[
\text{preço} = \beta_0 + \beta_1 \text{área} + \beta_2 \text{quartos} + \beta_3 \text{idade} + \varepsilon
\]

Os dados serão simulados para fins didáticos.

Além do ajuste do modelo, calcularemos o **VIF** (*Variance Inflation Factor*) para avaliar multicolinearidade entre os preditores.

In [ ]:
# Dados simulados: preço de imóvel
np.random.seed(42)

n = 100
area = np.random.uniform(40, 200, n)
quartos = np.random.randint(1, 5, n)
idade = np.random.uniform(1, 30, n)

preco = (
    150000
    + 1200 * area
    + 25000 * quartos
    - 3000 * idade
    + np.random.normal(0, 20000, n)
)

df = pd.DataFrame({
    'preco': preco,
    'area': area,
    'quartos': quartos,
    'idade': idade
})

df.head()

In [ ]:
X = sm.add_constant(df[['area', 'quartos', 'idade']])
modelo_mult = sm.OLS(df['preco'], X).fit()

print(modelo_mult.summary())

In [ ]:
# Cálculo do VIF
vif_df = pd.DataFrame({
    'variavel': X.columns,
    'VIF': [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
})

vif_df

## Perguntas para interpretação

1. Qual variável possui maior efeito estimado sobre o preço?
2. Como interpretar o coeficiente da variável `idade`?
3. O modelo parece globalmente significativo pelo teste $F$?
4. Há indícios de multicolinearidade preocupante pelo VIF?
5. Por que o VIF do intercepto não costuma ser interpretado substantivamente?

# 6. Exemplo completo: tempo de entrega

Neste exemplo, modelamos o tempo de entrega em função de duas variáveis:

- distância percorrida, em km;
- número de paradas intermediárias.

O objetivo é ajustar uma regressão múltipla e interpretar os coeficientes parciais.

In [ ]:
distancia = np.array([100, 150, 200, 200, 250, 300, 300, 350])
paradas = np.array([1, 2, 1, 3, 2, 3, 1, 4])
tempo = np.array([3.2, 5.1, 4.8, 7.0, 6.5, 8.2, 6.0, 10.1])

entregas = pd.DataFrame({
    'distancia': distancia,
    'paradas': paradas,
    'tempo': tempo
})

entregas

In [ ]:
X_entrega = sm.add_constant(entregas[['distancia', 'paradas']])
resultado = sm.OLS(entregas['tempo'], X_entrega).fit()

print(resultado.summary())

In [ ]:
b0, b_dist, b_paradas = resultado.params

print(f"Equação ajustada:")
print(f"tempo_hat = {b0:.3f} + {b_dist:.4f} * distancia + {b_paradas:.4f} * paradas")
print()
print(f"Interpretação da distância: mantendo o número de paradas constante,")
print(f"cada km adicional aumenta o tempo médio em {b_dist:.4f} horas.")
print()
print(f"Interpretação das paradas: mantendo a distância constante,")
print(f"cada parada adicional aumenta o tempo médio em {b_paradas:.4f} horas.")

In [ ]:
# Previsão para uma nova entrega
nova_entrega = pd.DataFrame({
    'const': [1],
    'distancia': [275],
    'paradas': [2]
})

pred = resultado.get_prediction(nova_entrega).summary_frame(alpha=0.05)
pred[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']]

## Atividade

Use o modelo anterior para responder:

1. Qual seria o tempo médio previsto para uma entrega de 275 km com 2 paradas?
2. Qual intervalo você usaria para comunicar a incerteza sobre a média?
3. Qual intervalo você usaria para comunicar a incerteza sobre uma entrega individual?
4. A distância e o número de paradas são significativos individualmente?
5. O modelo é globalmente significativo?

# 7. Exercício extra: criar seu próprio exemplo

Escolha um conjunto de dados com uma variável resposta quantitativa e pelo menos uma variável explicativa quantitativa.

Sugestões:

- altura e peso;
- tempo de estudo e nota;
- área e preço de imóvel;
- idade do carro e preço;
- distância e tempo de deslocamento.

Em seguida, siga o roteiro:

1. construa o diagrama de dispersão;
2. calcule a correlação;
3. ajuste uma regressão simples;
4. interprete os coeficientes;
5. avalie $R^2$;
6. construa gráficos de resíduos;
7. discuta limitações do modelo.

In [ ]:
# Modelo de roteiro para o exercício extra.
# Substitua os vetores abaixo pelos seus próprios dados.

x_ex = np.array([1, 2, 3, 4, 5])
y_ex = np.array([2.1, 2.9, 4.2, 5.1, 5.8])

# 1. Gráfico de dispersão
plt.figure(figsize=(5, 4))
plt.scatter(x_ex, y_ex)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Diagrama de dispersão")
plt.tight_layout()
plt.show()

# 2. Correlação
r_ex, p_ex = pearsonr(x_ex, y_ex)
print(f"r = {r_ex:.4f}, p-valor = {p_ex:.4f}")

# 3. Regressão simples
X_ex = sm.add_constant(x_ex)
modelo_ex = sm.OLS(y_ex, X_ex).fit()

print(modelo_ex.summary())

# 8. Fechamento

Este notebook cobre a parte computacional das notas de aula. Ao remover a seção “Implementação em Python” do material textual, os códigos passam a funcionar como um recurso prático independente, mais adequado para exploração pelos alunos.

Uma boa forma de usar este notebook em aula é pedir que os estudantes:

1. executem as células originais;
2. alterem os dados;
3. comparem correlação e inclinação;
4. observem o efeito de outliers;
5. comparem intervalo de confiança e intervalo de predição;
6. diagnostiquem resíduos em diferentes cenários.